## LINCS Level 5 데이터 전처리

### 목표

33GB 이상인 LINCS Level 5 `.gctx` 파일 전체를 메모리에 올리지 않고, metadata를 이용해 사용할 signature와 gene을 먼저 선택한 뒤 필요한 부분만 읽는다.

다음과 같은 supervised learning용 데이터를 만든다.

- 입력 `X`: drug ID, cell line, dose, treatment time
- 출력 `y`: 978개 landmark gene의 Level 5 signature value

##### 사용하는 파일

- `siginfo_beta.txt`
- `geneinfo_beta.txt`
- `compoundinfo_beta.txt`
- `level5_beta.gctx`

##### 최종 산출물

- `X_versionA.csv`
- `metadata_versionA.csv`
- `y_versionA.parquet`

### 1. 라이브러리 


In [1]:
import os
import numpy as np
import pandas as pd

from cmapPy.pandasGEXpress.parse import parse

import warnings
warnings.filterwarnings("ignore")

### 2. 파일 경로 Config


In [2]:
DATA_DIR = "./data"

SIGINFO_PATH = os.path.join(DATA_DIR, "siginfo_beta.txt")
GENEINFO_PATH = os.path.join(DATA_DIR, "geneinfo_beta.txt")
COMPOUNDINFO_PATH = os.path.join(DATA_DIR, "compoundinfo_beta.txt")
GCTX_PATH = os.path.join(DATA_DIR, "level5.gctx")

## 3. Metadata 로딩

먼저 metadata 파일인 `siginfo`, `geneinfo`, `compoundinfo`만 읽는다.

전체 `.gctx` 파일은 아직 읽지 않는다. 33GB 이상이므로 metadata에서 사용할 sample과 gene을 먼저 고른 뒤 필요한 부분만 부분 로딩한다.

In [3]:
siginfo = pd.read_csv(SIGINFO_PATH, sep="	", low_memory=False)
geneinfo = pd.read_csv(GENEINFO_PATH, sep="	", low_memory=False)
compoundinfo = pd.read_csv(COMPOUNDINFO_PATH, sep="	", low_memory=False)

print("siginfo shape:", siginfo.shape)
print("geneinfo shape:", geneinfo.shape)
print("compoundinfo shape:", compoundinfo.shape)

siginfo shape: (1201944, 37)
geneinfo shape: (12328, 7)
compoundinfo shape: (39321, 7)


In [4]:
print(f'siginfo columns : {siginfo.columns.tolist()}')

print(f'geneinfo columns :{geneinfo.columns.tolist()}')

print(f'compoundinfo columns : {compoundinfo.columns.tolist()}')

siginfo columns : ['bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig']
geneinfo columns :['gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space']
compoundinfo columns : ['pert_id', 'cmap_name', 'target', 'moa', 'canonical_smiles', 'inchi_key', 'compound_aliases']


### 4. 필요한 컬럼 확인


In [5]:
required_sig_cols = [
    "sig_id",
    "pert_type",
    "pert_id",
    "cell_iname",
    "pert_dose",
    "pert_time",
    "is_gold",
]

for col in required_sig_cols:
    if col in siginfo.columns:
        print(f'{col} : True')
    else:
        print(f'{col} : False')

sig_id : True
pert_type : True
pert_id : True
cell_iname : True
pert_dose : True
pert_time : True
is_gold : False


In [6]:
check_cols = ["pert_type", "cell_iname", "pert_time", "pert_dose", "is_gold"]

for col in check_cols:
    if col in siginfo.columns:

        print(f'[{col}] \n {siginfo[col].value_counts(dropna=False).head(20)}')

[pert_type] 
 pert_type
trt_cp             720216
trt_sh             177263
trt_xpr            140945
ctl_vehicle         39448
trt_sh.cgs          36720
trt_oe              34171
trt_sh.css          24368
ctl_vector          14969
trt_lig              7546
ctl_untrt            5332
trt_aby               575
trt_si                162
ctl_vector.cns        137
ctl_vehicle.cns        61
ctl_untrt.cns          30
ctl_x                   1
Name: count, dtype: int64
[cell_iname] 
 cell_iname
MCF7        110887
PC3         107579
A549        101214
A375         90241
HT29         83048
VCAP         73772
HA1E         67124
HEPG2        41087
HCC515       38477
YAPC         34046
MCF10A       30185
U2OS         26405
HELA         26170
MDAMB231     19976
NPC          19234
THP1         16979
HEK293       16635
ES2          15935
U251MG       15572
JURKAT       15197
Name: count, dtype: int64
[pert_time] 
 pert_time
 24.0     592155
 96.0     374798
 6.0      145491
 120.0     33200
 144.0    

### 5. Compound perturbation만 선택

약물 처리 perturbation만 사용한다.

따라서 `pert_type == "trt_cp"` 조건만 남긴다.

In [7]:
df_sig = siginfo.copy()

df_sig = df_sig[df_sig["pert_type"] == "trt_cp"].copy()

print("After pert_type filtering:", df_sig.shape)
print("unique drugs:", df_sig["pert_id"].nunique())
print("unique cells:", df_sig["cell_iname"].nunique())

After pert_type filtering: (720216, 37)
unique drugs: 34418
unique cells: 230


### 6. Treatment time 정리 및 24시간만 선택

실험에서는 treatment time을 24시간으로 고정하여 time에 따른 변동성을 줄이고 baseline 모델을 단순하게 만든다

In [8]:
df_sig["pert_time_num"] = pd.to_numeric(df_sig["pert_time"], errors="coerce")

print(df_sig["pert_time_num"].value_counts(dropna=False).head(20))

pert_time_num
 24.0     559163
 6.0      138006
 3.0       11998
 4.0        4704
 48.0       4372
 72.0        796
 12.0        360
-666.0       230
 2.0         188
 1.0         129
 120.0       124
 96.0        115
 336.0         6
 240.0         6
 288.0         6
 0.5           6
 168.0         6
 144.0         1
Name: count, dtype: int64


In [9]:
df_sig = df_sig[df_sig["pert_time_num"] == 24].copy()

print("After 24h filtering:", df_sig.shape)
print(df_sig["pert_time_num"].value_counts())

After 24h filtering: (559163, 38)
pert_time_num
24.0    559163
Name: count, dtype: int64


### 7. Dose 정리

`pert_dose`를 numeric 값으로 변환한다.(변환이 불가능한 값이나 결측값은 제거한다)

In [10]:
df_sig["pert_dose_num"] = pd.to_numeric(df_sig["pert_dose"], errors="coerce")

print("Dose NaN count:", df_sig["pert_dose_num"].isna().sum())
print(df_sig["pert_dose_num"].describe())

Dose NaN count: 56
count    559107.000000
mean          6.898942
std         619.482833
min           0.000041
25%           0.156250
50%           1.116340
75%          10.000000
max      316228.000000
Name: pert_dose_num, dtype: float64


In [11]:
df_sig = df_sig.dropna(subset=["pert_dose_num"]).copy()

print("After dose NaN removal:", df_sig.shape)

After dose NaN removal: (559107, 39)


### 8. 상위 cell line 선택

전체 cell line을 모두 쓰지 않고 sample 수가 많은 상위 cell line만 사용한다.


In [12]:
TOP_N_CELLS = 3

cell_counts = df_sig["cell_iname"].value_counts()
top_cells = cell_counts.head(TOP_N_CELLS).index.tolist()

print(f'Selected top cells:{top_cells}')

print(cell_counts.head(TOP_N_CELLS))

Selected top cells:['MCF7', 'PC3', 'A549']
cell_iname
MCF7    48327
PC3     46021
A549    44162
Name: count, dtype: int64


In [13]:
df_sig = df_sig[df_sig["cell_iname"].isin(top_cells)].copy()

print(f'After cell filtering: {df_sig.shape}')
print(df_sig["cell_iname"].value_counts())

After cell filtering: (138510, 39)
cell_iname
MCF7    48327
PC3     46021
A549    44162
Name: count, dtype: int64


### 9. 상위 drug 선택

전체 drug를 모두 쓰지 않고 sample 수가 많은 상위 drug만 사용한다.

In [14]:
TOP_N_DRUGS = 500

drug_counts = df_sig["pert_id"].value_counts()
top_drugs = drug_counts.head(TOP_N_DRUGS).index.tolist()

print("Number of selected drugs:", len(top_drugs))
print("Top drug counts:")
print(drug_counts.head(10))

Number of selected drugs: 500
Top drug counts:
pert_id
BRD-K60230970    2229
BRD-K50691590    1931
BRD-K81418486     634
BRD-A19500257     370
BRD-A75409952     354
BRD-K88510285     319
BRD-A19037878     315
BRD-K21680192     271
BRD-K88378636     235
BRD-A13084692     190
Name: count, dtype: int64


In [15]:
df_sig = df_sig[df_sig["pert_id"].isin(top_drugs)].copy()

print(f'After drug filtering : {df_sig.shape}')

After drug filtering : (35149, 39)


### 10. 최종 signature ID 목록 생성

`.gctx` 파일에서 읽어올 column id 목록을 만든다.

Level 5 `.gctx`에서 column은 signature id에 해당한다.

In [16]:
selected_sig_ids = df_sig["sig_id"].astype(str).unique().tolist()

len(selected_sig_ids)

35149

### 11. Landmark gene 선택

 978개 landmark gene만 target으로 사용한다.

`geneinfo_beta.txt`에서 landmark gene을 고른다.

In [17]:
geneinfo.head()

,gene_id,gene_symbol,ensembl_id,gene_title,gene_type,src,feature_space
0,750,GAS8-AS1,ENSG00000221819,GAS8 antisense RNA 1,ncRNA,NCBI,inferred
1,6315,ATXN8OS,NaN,ATXN8 opposite strand lncRNA,ncRNA,NCBI,inferred
2,7503,XIST,ENSG00000229807,X inactive specific transcript,ncRNA,NCBI,inferred
3,8552,INE1,ENSG00000224975,inactivation escape 1,ncRNA,NCBI,inferred
4,9834,FAM30A,ENSG00000226777,family with sequence similarity 30 member A,ncRNA,NCBI,inferred


In [18]:
for col in geneinfo.columns:
    print(f'[{col}]\n {geneinfo[col].dropna().astype(str).value_counts().head(10)}')

[gene_id]
 gene_id
750      1
6315     1
7503     1
8552     1
9834     1
10141    1
10255    1
10281    1
23434    1
23591    1
Name: count, dtype: int64
[gene_symbol]
 gene_symbol
MIA2         2
GAS8-AS1     1
ATXN8OS      1
XIST         1
INE1         1
FAM30A       1
LINC01587    1
HCG9         1
DSCR4        1
LINC01565    1
Name: count, dtype: int64
[ensembl_id]
 ensembl_id
ENSG00000174640    2
ENSG00000221819    1
ENSG00000229807    1
ENSG00000224975    1
ENSG00000226777    1
ENSG00000082929    1
ENSG00000204625    1
ENSG00000184029    1
ENSG00000198685    1
ENSG00000267496    1
Name: count, dtype: int64
[gene_title]
 gene_title
GAS8 antisense RNA 1                            1
ATXN8 opposite strand lncRNA                    1
X inactive specific transcript                  1
inactivation escape 1                           1
family with sequence similarity 30 member A     1
long intergenic non-protein coding RNA 1587     1
HLA complex group 9                             1
Down s

In [19]:
landmark_genes = geneinfo[geneinfo["feature_space"] == "landmark"].copy()

print(f'landmark_genes shape: {landmark_genes.shape}')
landmark_genes.head()

landmark_genes shape: (978, 7)


,gene_id,gene_symbol,ensembl_id,gene_title,gene_type,src,feature_space
2154,16,AARS,ENSG00000090861,alanyl-tRNA synthetase,protein-coding,NCBI,landmark
2155,23,ABCF1,ENSG00000204574,ATP binding cassette subfamily F member 1,protein-coding,NCBI,landmark
2156,25,ABL1,ENSG00000097007,"ABL proto-oncogene 1, non-receptor tyrosine ki...",protein-coding,NCBI,landmark
2157,30,ACAA1,ENSG00000060971,acetyl-CoA acyltransferase 1,protein-coding,NCBI,landmark
2158,39,ACAT2,ENSG00000120437,acetyl-CoA acetyltransferase 2,protein-coding,NCBI,landmark


In [20]:
selected_gene_ids = landmark_genes["gene_id"].astype(str).unique().tolist()

selected_gene_ids[:10]

['16', '23', '25', '30', '39', '47', '102', '128', '142', '154']

### 12. `.gctx` 부분 로딩

33GB 전체 파일을 읽지 않고 선택한 gene row와 signature column만 읽는다.

- `rid`: gene id 목록
- `cid`: signature id 목록

In [21]:
gctoo = parse(
    GCTX_PATH,
    rid=selected_gene_ids,
    cid=selected_sig_ids,
)

expr = gctoo.data_df

print(f'Loaded expression shape:{expr.shape}')

Loaded expression shape:(978, 35149)


### 13. Expression matrix transpose

In [22]:
expr = expr.T.copy()

expr.index = expr.index.astype(str)
expr.columns = expr.columns.astype(str)

print(f'Expression shape after transpose:{expr.shape}')
# rows = signatures, columns = genes
expr.head()

Expression shape after transpose:(35149, 978)


rid,10007,1001,10013,10038,10046,10049,10051,10057,10058,10059,...,9918,9924,9926,9928,993,994,9943,9961,998,9988
cid,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-A61304759:0.625:24,0.267136,-0.456491,2.165093,0.229504,-0.124224,1.576591,-1.828272,-0.225401,1.707536,-0.433627,...,1.564901,0.421255,-2.748089,-2.485454,-3.207149,-5.228687,-0.181523,-1.382612,-1.131088,1.328648
ABY001_A549_XH:BRD-A61304759:10:24,-0.117392,0.200503,3.915682,0.172298,-0.620917,1.896570,-0.351663,2.388200,2.241555,0.882148,...,2.835503,-2.048177,-1.072422,-1.558214,-3.306095,-1.125444,-0.274837,-2.573916,-1.041967,0.582418
ABY001_A549_XH:BRD-A61304759:2.5:24,0.703145,-0.459059,2.882389,-0.590942,-0.809936,1.652521,-1.464148,1.901619,1.833337,-0.166399,...,2.853060,-1.136871,-1.968534,-1.382610,-3.071453,-2.076343,-0.108275,-3.197631,0.172597,0.360270
ABY001_A549_XH:BRD-A90490067:0.625:24,0.911685,-0.032964,-0.585127,-0.394690,-0.081405,0.156198,1.680390,0.157106,-1.150557,-0.833759,...,-1.232962,-1.763807,0.304337,0.471825,-0.021767,-0.316735,0.263729,-0.352919,0.556488,0.320331
ABY001_A549_XH:BRD-A90490067:10:24,1.412661,0.352456,0.386300,-1.181262,3.863513,0.755301,-1.086625,-0.230203,-1.392281,-3.727853,...,-0.482366,-0.636021,-1.075500,-1.617027,-1.222583,-0.952263,-1.018351,0.424994,-0.618140,1.483994


### 14. Metadata와 expression matrix 정렬

metadata의 signature id와 expression matrix의 signature id를 맞춘다. (입력 데이터와 출력의 sample 순서가 다르면 모델 학습 결과가 무의미해지기 때문)

In [23]:
df_meta = df_sig.copy()
df_meta["sig_id"] = df_meta["sig_id"].astype(str)
df_meta = df_meta.set_index("sig_id")

# print(df_meta.head(1))

common_sig_ids = expr.index.intersection(df_meta.index)

print(f'expr signatures:{expr.shape[0]}')
print(f'metadata signatures:{df_meta.shape[0]}')
print(f'common signatures:{len(common_sig_ids)}')

expr signatures:35149
metadata signatures:35149
common signatures:35149


In [24]:
expr = expr.loc[common_sig_ids].copy()
df_meta = df_meta.loc[common_sig_ids].copy()

df_meta = df_meta.loc[expr.index].copy()

print(f'Aligned expr shape:{expr.shape}')
print(f'Aligned metadata shape:{df_meta.shape}')

Aligned expr shape:(35149, 978)
Aligned metadata shape:(35149, 38)


In [25]:
'''compound_cols = [
    "pert_id",
    "cmap_name",
    "target",
    "moa",
    "canonical_smiles",
    "inchi_key",
]

compound_meta = compoundinfo[compound_cols].copy()

compound_meta['pert_id'] = compound_meta["pert_id"].astype(str)
df_meta["pert_id"] = df_meta["pert_id"].astype(str)

df_meta.index.name = "sig_id"

df_meta = (df_meta.reset_index().merge(compound_meta, on = 'pert_id', how = 'left').set_index("sig_id"))

print(df_meta.shape)
print(df_meta.head())'''


'compound_cols = [\n    "pert_id",\n    "cmap_name",\n    "target",\n    "moa",\n    "canonical_smiles",\n    "inchi_key",\n]\n\ncompound_meta = compoundinfo[compound_cols].copy()\n\ncompound_meta[\'pert_id\'] = compound_meta["pert_id"].astype(str)\ndf_meta["pert_id"] = df_meta["pert_id"].astype(str)\n\ndf_meta.index.name = "sig_id"\n\ndf_meta = (df_meta.reset_index().merge(compound_meta, on = \'pert_id\', how = \'left\').set_index("sig_id"))\n\nprint(df_meta.shape)\nprint(df_meta.head())'

### 15. 결측값 확인 및 제거

In [26]:
feature_cols = ["pert_id", "cell_iname", "pert_dose_num", "pert_time_num"]

common_ids  = df_meta.index.intersection(expr.index)

df_meta = df_meta.loc[common_ids].copy()
expr = expr.loc[common_ids].copy()

expr = expr.loc[df_meta.index].copy()

print(df_meta[feature_cols].isna().sum())
print(expr.isna().sum().sum())

print(f"[before]\n {df_meta.shape} \n {expr.shape}")

pert_id          0
cell_iname       0
pert_dose_num    0
pert_time_num    0
dtype: int64
0
[before]
 (35149, 38) 
 (35149, 978)


In [27]:
valid_meta_mask = ~df_meta[feature_cols].isna().any(axis=1)
valid_expr_mask = ~expr.isna().any(axis=1)

valid_ids = df_meta.index[valid_meta_mask & valid_expr_mask]

df_meta = df_meta.loc[valid_ids].copy()
expr = expr.loc[valid_ids].copy()

print(f"[After]\n {df_meta.shape} \n {expr.shape}")

[After]
 (35149, 38) 
 (35149, 978)


### 16. 최종 X, y 생성

`X`: pert_id,cell_iname,pert_dose_num,pert_time_num

`y`: 978 landmark gene signature values

In [28]:
X = df_meta[feature_cols].copy()
y = expr.copy()

print(f'X shape:{X.shape}')
print(f'y shape:{y.shape}')

display(X.head())
display(y.head())

X shape:(35149, 4)
y shape:(35149, 978)


,pert_id,cell_iname,pert_dose_num,pert_time_num
ABY001_A549_XH:BRD-A61304759:0.625:24,BRD-A61304759,A549,0.625,24.0
ABY001_A549_XH:BRD-A61304759:10:24,BRD-A61304759,A549,10.000,24.0
ABY001_A549_XH:BRD-A61304759:2.5:24,BRD-A61304759,A549,2.500,24.0
ABY001_A549_XH:BRD-A90490067:0.625:24,BRD-A90490067,A549,0.625,24.0
ABY001_A549_XH:BRD-A90490067:10:24,BRD-A90490067,A549,10.000,24.0


rid,10007,1001,10013,10038,10046,10049,10051,10057,10058,10059,...,9918,9924,9926,9928,993,994,9943,9961,998,9988
ABY001_A549_XH:BRD-A61304759:0.625:24,0.267136,-0.456491,2.165093,0.229504,-0.124224,1.576591,-1.828272,-0.225401,1.707536,-0.433627,...,1.564901,0.421255,-2.748089,-2.485454,-3.207149,-5.228687,-0.181523,-1.382612,-1.131088,1.328648
ABY001_A549_XH:BRD-A61304759:10:24,-0.117392,0.200503,3.915682,0.172298,-0.620917,1.896570,-0.351663,2.388200,2.241555,0.882148,...,2.835503,-2.048177,-1.072422,-1.558214,-3.306095,-1.125444,-0.274837,-2.573916,-1.041967,0.582418
ABY001_A549_XH:BRD-A61304759:2.5:24,0.703145,-0.459059,2.882389,-0.590942,-0.809936,1.652521,-1.464148,1.901619,1.833337,-0.166399,...,2.853060,-1.136871,-1.968534,-1.382610,-3.071453,-2.076343,-0.108275,-3.197631,0.172597,0.360270
ABY001_A549_XH:BRD-A90490067:0.625:24,0.911685,-0.032964,-0.585127,-0.394690,-0.081405,0.156198,1.680390,0.157106,-1.150557,-0.833759,...,-1.232962,-1.763807,0.304337,0.471825,-0.021767,-0.316735,0.263729,-0.352919,0.556488,0.320331
ABY001_A549_XH:BRD-A90490067:10:24,1.412661,0.352456,0.386300,-1.181262,3.863513,0.755301,-1.086625,-0.230203,-1.392281,-3.727853,...,-0.482366,-0.636021,-1.075500,-1.617027,-1.222583,-0.952263,-1.018351,0.424994,-0.618140,1.483994


### 17. 최종 데이터 요약

In [29]:
summary = {
    "n_samples": X.shape[0],
    "n_features_raw": X.shape[1],
    "n_target_genes": y.shape[1],
    "n_drugs": X["pert_id"].nunique(),
    "n_cells": X["cell_iname"].nunique(),
    "dose_min": X["pert_dose_num"].min(),
    "dose_max": X["pert_dose_num"].max(),
    "time_values": sorted(X["pert_time_num"].unique().tolist()),
}

summary

{'n_samples': 35149,
 'n_features_raw': 4,
 'n_target_genes': 978,
 'n_drugs': 500,
 'n_cells': 3,
 'dose_min': np.float64(0.000949668),
 'dose_max': np.float64(177.6),
 'time_values': [24.0]}

In [30]:
print(f'Cell line distribution\n {X["cell_iname"].value_counts()}')

print(f'Top drug distribution" \n{X["pert_id"].value_counts().head(20)}')

Cell line distribution
 cell_iname
MCF7    12629
PC3     11492
A549    11028
Name: count, dtype: int64
Top drug distribution" 
pert_id
BRD-K60230970    2229
BRD-K50691590    1931
BRD-K81418486     634
BRD-A19500257     370
BRD-A75409952     354
BRD-K88510285     319
BRD-A19037878     315
BRD-K21680192     271
BRD-K88378636     235
BRD-A13084692     190
BRD-A61304759     185
BRD-A79768653     182
BRD-K49328571     172
BRD-K49865102     170
BRD-K59369769     158
BRD-K75295174     151
BRD-K63828191     142
BRD-K27305650     134
BRD-K77908580     132
BRD-K12184916     131
Name: count, dtype: int64


### 18. 전처리된 데이터 저장

In [31]:
OUTPUT_DIR = "./lincs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X.to_csv(os.path.join(OUTPUT_DIR, "X.csv"), index=True)
df_meta.to_csv(os.path.join(OUTPUT_DIR, "metadata_X.csv"), index=True)

y.to_parquet(os.path.join(OUTPUT_DIR, "y.parquet"))

In [32]:
X_loaded = pd.read_csv(
    os.path.join(OUTPUT_DIR, "X.csv"),
    index_col=0,
)

meta_loaded = pd.read_csv(
    os.path.join(OUTPUT_DIR, "metadata_X.csv"),
    index_col=0,
)

y_loaded = pd.read_parquet(
    os.path.join(OUTPUT_DIR, "y.parquet"))

print(X_loaded.shape)
print(meta_loaded.shape)
print(y_loaded.shape)

(35149, 4)
(35149, 38)
(35149, 978)


## 최종 데이터

전처리 후 다음 객체를 사용하면 된다.

```python
X        # 입력 feature: drug, cell, dose, time
y        # target: 978 landmark gene Level 5 signature
df_meta  # signature metadata
```